# Sequential RGB SteganoGAN — PROVEN FINAL VERSION
**Loss: MSE(image quality) + BCE(decoder accuracy) — same as original SteganoGAN paper**



In [ ]:
!pip install -q reedsolo tqdm Pillow imageio
import os, zipfile, requests
LOCAL = '/kaggle/working/data'; os.makedirs(LOCAL, exist_ok=True)
def dl(url, fn):
    if not os.path.exists(fn):
        print(f'Downloading {fn}...'); r = requests.get(url, stream=True)
        with open(fn, 'wb') as f:
            for chunk in r.iter_content(1024*1024): f.write(chunk)
    else: print(f'Skipping {fn}')
dl('http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip', f'{LOCAL}/train.zip')
dl('http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip', f'{LOCAL}/val.zip')
if not os.path.exists(f'{LOCAL}/DIV2K_train_HR'):
    print('Extracting...'); zipfile.ZipFile(f'{LOCAL}/train.zip').extractall(LOCAL); zipfile.ZipFile(f'{LOCAL}/val.zip').extractall(LOCAL)
print(f'Train: {len(os.listdir(f"{LOCAL}/DIV2K_train_HR"))} | Val: {len(os.listdir(f"{LOCAL}/DIV2K_valid_HR"))}')

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print(f'GPU: {torch.cuda.get_device_name(0)} | Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

In [ ]:
import os, glob, json, torch, numpy as np
from PIL import Image
from math import exp, log10
from tqdm.auto import tqdm
from torch import nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torch.nn.functional import (
    conv2d, mse_loss,
    binary_cross_entropy_with_logits as bce  # CORRECT loss for binary payload
)

def ssim(i1, i2, ws=11):
    def gaussian(ws, s):
        e = [exp(-(x-ws//2)**2/float(2*s**2)) for x in range(ws)]
        g = torch.Tensor(e); return g/g.sum()
    ch = i1.size(1)
    w = gaussian(ws,1.5).unsqueeze(1).mm(gaussian(ws,1.5).unsqueeze(1).t()).float().unsqueeze(0).unsqueeze(0).expand(ch,1,ws,ws).to(i1.device)
    m1,m2 = conv2d(i1,w,padding=ws//2,groups=ch), conv2d(i2,w,padding=ws//2,groups=ch)
    m1s,m2s,m12 = m1**2,m2**2,m1*m2
    s1 = conv2d(i1*i1,w,padding=ws//2,groups=ch)-m1s
    s2 = conv2d(i2*i2,w,padding=ws//2,groups=ch)-m2s
    s12 = conv2d(i1*i2,w,padding=ws//2,groups=ch)-m12
    C1,C2 = 0.01**2, 0.03**2
    return ((2*m12+C1)*(2*s12+C2)/((m1s+m2s+C1)*(s1+s2+C2))).mean()

# Encoder — matches local steganogan/encoders.py ResidualEncoder exactly
class ResidualEncoder(nn.Module):
    def __init__(self, data_depth, hidden_size):
        super().__init__()
        self.data_depth = data_depth
        self.hidden_size = hidden_size
        self.features = nn.Sequential(
            nn.Conv2d(1, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
        )
        self.layers = nn.Sequential(
            nn.Conv2d(hidden_size + data_depth, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
            nn.Conv2d(hidden_size, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
            nn.Conv2d(hidden_size, 1, 3, padding=1),
        )
    def forward(self, image, data):
        x = self.features(image)
        x = self.layers(torch.cat([x, data], dim=1))
        return image + x  # residual: output = cover + tiny_change

# Decoder — matches local steganogan/decoders.py
class DenseDecoder(nn.Module):
    def __init__(self, data_depth, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
            nn.Conv2d(hidden_size, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
            nn.Conv2d(hidden_size, hidden_size, 3, padding=1),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(hidden_size),
            nn.Conv2d(hidden_size, data_depth, 3, padding=1),
        )
    def forward(self, image): return self.layers(image)  # outputs logits

class SteganoDataset(Dataset):
    def __init__(self, path, limit=None):
        self.files = sorted(glob.glob(os.path.join(path, '*.png')))
        if limit: self.files = self.files[:limit]
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB').resize((256, 256))
        return torch.FloatTensor(np.array(img)/127.5-1.0).permute(2,0,1), 0

class SteganoGAN(nn.Module):
    def __init__(self, data_depth=1, hidden_size=32):
        super().__init__()
        self.data_depth = data_depth
        self.dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.encoder = ResidualEncoder(data_depth, hidden_size).to(self.dev)
        self.decoder = DenseDecoder(data_depth, hidden_size).to(self.dev)
        self.opt = Adam(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=1e-4
        )

    def _encode(self, cover, payload):
        # Sequential RGB: each channel independently
        dd = self.data_depth
        sR = self.encoder(cover[:,0:1], payload[:,0:dd])
        sG = self.encoder(cover[:,1:2], payload[:,dd:2*dd])
        sB = self.encoder(cover[:,2:3], payload[:,2*dd:3*dd])
        return torch.cat([sR, sG, sB], dim=1)

    def _decode(self, stego):
        return torch.cat([
            self.decoder(stego[:,0:1]),
            self.decoder(stego[:,1:2]),
            self.decoder(stego[:,2:3])
        ], dim=1)

    def fit(self, tl, vl, epochs=50):
        all_metrics = []
        for e in range(1, epochs + 1):
            self.train()
            for cover, _ in tqdm(tl, desc=f'Epoch {e}/{epochs}'):
                cover = cover.to(self.dev)
                N, _, H, W = cover.size()
                # payload is float 0.0 or 1.0 for BCE
                payload = torch.zeros(N, self.data_depth*3, H, W).random_(0, 2).float().to(self.dev)

                self.opt.zero_grad()
                gen = self._encode(cover, payload)
                dec = self._decode(gen)  # raw logits

                # Proven stable loss:
                # - mse_loss(gen, cover): keeps stego visually identical to cover
                # - bce(dec, payload): correct loss for binary data (not MSE!)
                # Equal weights — no competition, both improve together
                loss = mse_loss(gen, cover) + bce(dec, payload)
                loss.backward()
                self.opt.step()
                del gen, dec; torch.cuda.empty_cache()

            self.eval()
            with torch.no_grad():
                ps, ss, ac, cn = 0, 0, 0, 0
                for cv, _ in vl:
                    cv = cv.to(self.dev)
                    N, _, H, W = cv.size()
                    pv = torch.zeros(N, self.data_depth*3, H, W).random_(0, 2).float().to(self.dev)
                    gv = self._encode(cv, pv)
                    dv = self._decode(gv)  # logits
                    mse = mse_loss(gv, cv).item()
                    ps += 10 * log10(4 / max(mse, 1e-10))
                    ss += ssim(cv, gv).item()
                    # accuracy: logit > 0 means predicted 1 (correct for BCE)
                    ac += ((dv > 0).float() == pv).float().mean().item()
                    cn += 1
                m = {'epoch': e, 'psnr': round(ps/cn,2), 'ssim': round(ss/cn,4), 'acc': round(ac/cn,4)}
                all_metrics.append(m)
                print(f"Epoch {e:02d}: PSNR={m['psnr']:.2f} dB | SSIM={m['ssim']:.4f} | Acc={m['acc']:.4f}")

        torch.save(self.state_dict(), '/kaggle/working/sequential_rgb_FINAL.steg')
        with open('/kaggle/working/metrics.json', 'w') as f: json.dump(all_metrics, f, indent=2)
        print('\n✅ Model saved → sequential_rgb_FINAL.steg')
        print('✅ Metrics saved → metrics.json')
        return all_metrics

# ---- Run ----
model = SteganoGAN(data_depth=1, hidden_size=32)
tl = DataLoader(SteganoDataset('/kaggle/working/data/DIV2K_train_HR'), batch_size=4, shuffle=True)
vl = DataLoader(SteganoDataset('/kaggle/working/data/DIV2K_valid_HR', limit=20), batch_size=4, shuffle=False)
print(f'Train: {len(tl.dataset)} | Val: {len(vl.dataset)} | Device: {model.dev}')
model.fit(tl, vl, epochs=50)

In [ ]:
import json
with open('/kaggle/working/metrics.json') as f: m = json.load(f)
print(f"{'Epoch':<8}{'PSNR (dB)':<14}{'SSIM':<12}{'Decode Acc':<12}")
print('-'*46)
for r in m:
    print(f"{r['epoch']:<8}{r['psnr']:<14.2f}{r['ssim']:<12.4f}{r['acc']:<12.4f}")
print('\n✅ Paste into your Final Year Project Report!')